# Einops, Einsum & Tensor Manipulation

In [2]:
import einops
import numpy as np
import plotly.express as px
import torch as t

def plotim(img_array):
    """
    Displays a numpy array as an image

    Two options:
        img_array.shape = (height, width) -> interpreted as monochrome
        img_array.shape = (3, height, width) -> interpreted as RGB
    """
    shape = img_array.shape
    assert len(shape) == 2 or (shape[0] == 3 and len(shape) == 3), "Incorrect format (see docstring)"

    if len(shape) == 3:
        img_array = einops.rearrange(img_array, "c h w -> h w c")
    height, width = img_array.shape[:2]

    fig = px.imshow(img_array, zmin=0, zmax=255, color_continuous_scale="gray")
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_layout(coloraxis_showscale=False, margin=dict.fromkeys("tblr", 0), height=height, width=width)
    fig.show(config=dict(displayModeBar=False))


## Einops

In [3]:
arr = np.load("numbers.npy")
print(arr.shape)
plotim(arr[5])

(6, 3, 150, 150)


In [4]:
arr2 = einops.rearrange(arr, "b c h w -> c h (b w)")
print(arr2.shape)
plotim(arr2)

(3, 150, 900)


In [5]:
arr2 = einops.repeat(arr[5],"c h w -> c h ( 3 w)")
print(arr2.shape)
plotim(arr2)

(3, 150, 450)


In [6]:
arr2 = einops.repeat(arr[0:2], "b c h w ->  c (b h) (3 w )")
print(arr2.shape)
plotim(arr2)

(3, 300, 450)


In [7]:
arr2 = einops.repeat(arr[5],"c h w  -> c h (w 3)" )


In [ ]:
arr2 = einops.rearrange(arr[5],"c h w -> c w h") # matriz traspuesta
print(arr2.shape)
plotim(arr2)

(3, 150, 150)


In [33]:
a  = t.arange(1,7)
o = einops.repeat(a,'x -> (3 x)') # tomamos el x y lo repetimos 3 vece
i = einops.repeat(a,'x -> (x 3)') # tomamos cada elemento de x y lo repetimos tres vecesa
k = einops.reduce(a,'(x 2) -> x',"max") # x=3 e indica fila
l = einops.reduce(a,'(2 x) -> x',"max") # x=3 e indica columna
print(a.unsqueeze(1).shape)
print(t.arange(3,9).unsqueeze(1).shape)
m = einops.rearrange(a.unsqueeze(1), "(f b1) c -> f (b1 c)",b1=2)
n = einops.rearrange(a, "(f c)  -> f c",f=2,c=3)
print('Tomamos x y lo repetimos 3 veces:', o)
print('Tomamos cada elemento de x y lo repetimos 3 veces:',i)
print('Dividimos a en tres grupos de 2 y hallamos el máximo en cada x',k)
print('Dividimos a dos grupos de 3 y hallamos el máximo en cada x',l)
print('Convertimos un vector de 6 elementos en una matriz de 3x2:',m)
print('Convertimos un vector de 6 elementos en una matriz de 2x3:',n)

torch.Size([6, 1])
torch.Size([6, 1])
Tomamos x y lo repetimos 3 veces: tensor([1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6])
Tomamos cada elemento de x y lo repetimos 3 veces: tensor([1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6])
Dividimos a en tres grupos de 2 y hallamos el máximo en cada x tensor([2, 4, 6])
Dividimos a dos grupos de 3 y hallamos el máximo en cada x tensor([4, 5, 6])
Convertimos un vector de 6 elementos en una matriz de 3x2: tensor([[1, 2],
        [3, 4],
        [5, 6]])
Convertimos un vector de 6 elementos en una matriz de 2x3: tensor([[1, 2, 3],
        [4, 5, 6]])


In [37]:
m = einops.rearrange(t.arange(1,10), '(f c) -> f c',f=3,c=3).float()
mn = m/m.norm(dim=1,keepdim=True)
print('Matriz:\n',m)
print('Matriz normalizada, es decir, cada elemento dividido por la norma  en dim=1\n',mn)

Matriz:
 tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
Matriz normalizada, es decir, cada elemento dividido por la norma  en dim=1
 tensor([[0.2673, 0.5345, 0.8018],
        [0.4558, 0.5698, 0.6838],
        [0.5026, 0.5744, 0.6462]])


## Broadcasting
Las reglas de broadcasting entre dos tensores son:
* Se pueden agregar dimensiones dummy de valor uno al comienzo de un tensor hasta que ambos tensores tengan la misma dimensión. 
* Después de esto si alguna dimensión tiene tamaño uno en uno de sus tensores, se repite hasta que se matchea el tamaño de la dimensión en el otro tensor. 
  
Eg. Tengo que sumar una matriz de dimensión A de dimensión (N,k) con un vector de dimensión B de dimensión (k). Primero agregamos una dimensión al vector B de modo que nos quede con dimensión (1,k). Luego repetimos N veces el vector k de modo que nos quede de dimensión (N,k). 

La forma de agregar dimensiones es mendiante el método `unsqueeze`. Si tengo un tensor de dimensión (3,6,8,4,2) podremos agregar una dimensión en la posición i haciendo `v.unsqueeze(i)`. Si i=2 nos quedará de dimensión (3,6,1,8,4,2) 


In [10]:
a = t.ones((3,6,8,4,2))
a.unsqueeze(2).shape

torch.Size([3, 6, 1, 8, 4, 2])

In [11]:
arr = t.arange(3,9).unsqueeze(1)
arr.shape
expected = t.tensor([[3, 4], [5, 6], [7, 8]])
expected.shape

torch.Size([3, 2])